# Crawl Pipeline — Corpus Gap Questions

**Trước khi chạy:**
1. Runtime → Change runtime type → **T4 GPU**
2. Thêm secrets (tab 🔑 bên trái):
   - `GROQ_API_KEY` = key từ console.groq.com
3. Upload file `gap_questions.jsonl` vào session (kéo thả vào file panel)


In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/callbot_crawl'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f'Output sẽ lưu tại: {DRIVE_OUTPUT_DIR}')

In [ ]:
# Cell 2 — Clone repo
import os

REPO_URL = 'https://github.com/enigmaticcat/legal-voice-callbot.git'
REPO_DIR = '/content/callbot'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Done')

In [ ]:
# Cell 3 — Install dependencies
!pip install -q groq ddgs trafilatura sentence-transformers python-dotenv

In [ ]:
# Cell 4 — Set API key từ Colab Secrets
import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('GROQ_API_KEY:', 'OK' if os.environ.get('GROQ_API_KEY') else 'MISSING')

In [ ]:
# Cell 5 — Upload gap_questions.jsonl
# Nếu chưa upload, chạy cell này để upload từ máy
# Nếu đã kéo thả vào file panel rồi thì bỏ qua

from google.colab import files
import shutil

uploaded = files.upload()  # chọn file gap_questions.jsonl
for fname in uploaded:
    shutil.move(fname, f'/content/callbot/{fname}')
    print(f'Moved: {fname} → /content/callbot/{fname}')

In [ ]:
# Cell 6 — Kiểm tra GPU và files
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import os
print(f'gap_questions.jsonl exists: {os.path.exists("/content/callbot/gap_questions.jsonl")}')

In [ ]:
# Cell 7 — Config: đổi output path sang Drive
# Patch config trước khi chạy pipeline

DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/callbot_crawl'

OUTPUT_JSONL   = f'{DRIVE_OUTPUT_DIR}/new_corpus_chunks.jsonl'
FAILED_LOG     = f'{DRIVE_OUTPUT_DIR}/crawl_failed.txt'
CHECKPOINT_DIR = f'{DRIVE_OUTPUT_DIR}/checkpoints'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Output  : {OUTPUT_JSONL}')
print(f'Failed  : {FAILED_LOG}')
print(f'Ckpt dir: {CHECKPOINT_DIR}')

In [ ]:
# Cell 8 — Chạy crawl pipeline
# Patch các đường dẫn output rồi exec pipeline

import builtins, types

# Override config constants trước khi pipeline load
crawl_src = open('/content/callbot/crawl_pipeline.py', encoding='utf-8').read()

crawl_src = crawl_src.replace(
    'OUTPUT_JSONL        = "new_corpus_chunks.jsonl"',
    f'OUTPUT_JSONL        = "{OUTPUT_JSONL}"'
).replace(
    'FAILED_LOG          = "crawl_failed.txt"',
    f'FAILED_LOG          = "{FAILED_LOG}"'
).replace(
    'CHECKPOINT_DIR      = "checkpoints"',
    f'CHECKPOINT_DIR      = "{CHECKPOINT_DIR}"'
).replace(
    'GAP_QUESTIONS_PATH  = "gap_questions.jsonl"',
    'GAP_QUESTIONS_PATH  = "/content/callbot/gap_questions.jsonl"'
)

exec(crawl_src, {'__name__': '__main__'})

In [ ]:
# Cell 9 — Kiểm tra kết quả
import json

if os.path.exists(OUTPUT_JSONL):
    lines = open(OUTPUT_JSONL, encoding='utf-8').readlines()
    print(f'Tổng docs crawled: {len(lines)}')
    if lines:
        sample = json.loads(lines[0])
        print(f'Sample source : {sample["source"]}')
        print(f'Sample score  : {sample["relevance_score"]}')
        print(f'Sample text   : {sample["text"][:200]}')
else:
    print('Output file chưa có — pipeline chưa chạy hoặc bị lỗi')